## Análise da utilização de meta-features baseadas em ELA para um sistema de recomendação de otimização de hiperparâmetros

Script destinado para identificar as melhores combinações do experimento por meio da aplicação de testes estatísticos não paramétricos, como teste de Friedman e Nemenyi post hoc.

### Bibliotecas

In [2]:
import os
import itertools
from collections import Counter
import numpy as np
import pandas as pd
import ast

from tqdm import tqdm 

from scipy import stats
import scikit_posthocs as sp

In [11]:
resultados_list = os.listdir("../resultados/default/resultados_combinacoes/")

In [ ]:
files = ['ela_features_flacco', 'classif_svm_169d_95_average', 'classif_svm_ela_features_flacco']

In [13]:
resultados = {"arquivo": [], "media_f1_score": [], "mediana_f1_score": [], "desvio_padrao_f1_score": [], "min_f1_score": [], "max_f1_score": [], "varianca_f1_score": [], "media_acc_balanceada": [], "mediana_acc_balanceada": [], "desvio_padrao_acc_balanceada": [], "min_acc_balanceada": [], "max_acc_balanceada": [], "varianca_acc_balanceada": [], "media_auc": [], "mediana_auc": [], "desvio_padrao_auc": [], "min_auc": [], "max_auc": [], "varianca_auc": [], "instancias_mais_erradas": []}

In [16]:
# analisa cada um dos arquivos e salva os resultados em um arquivo de resumo
for r in tqdm(resultados_list):
    dados = pd.read_csv("../resultados/default/resultados_combinacoes/" + r, sep = ';')
    resultados["arquivo"].append(r)
   
    seeds = np.unique(dados['seed'].values)

    medias_f1_score_seed = []
    medias_auc_seed = []
    medias_acc_balanceada_seed = []

    for s in seeds:
        medias_f1_score_seed.append(np.mean([dados['f1_score'][i] for i, j in enumerate(dados['seed']) if j == s]))
        medias_auc_seed.append(np.mean([dados['auc'][i] for i, j in enumerate(dados['seed']) if j == s]))
        medias_acc_balanceada_seed.append(np.mean([dados['acuracia_balanceada'][i] for i, j in enumerate(dados['seed']) if j == s]))

    resultados["desvio_padrao_f1_score"].append(np.std(medias_f1_score_seed))
    resultados["desvio_padrao_auc"].append( np.std(medias_auc_seed))
    resultados["desvio_padrao_acc_balanceada"].append(np.std(medias_acc_balanceada_seed))
    
    resultados["varianca_f1_score"].append(np.var(medias_f1_score_seed))
    resultados["varianca_auc"].append(np.var(medias_auc_seed))
    resultados["varianca_acc_balanceada"].append(np.var(medias_acc_balanceada_seed))

    resultados["min_f1_score"].append(min(medias_f1_score_seed))
    resultados["min_auc"].append(min(medias_auc_seed))
    resultados["min_acc_balanceada"].append(min(medias_acc_balanceada_seed))

    resultados["max_f1_score"].append(max(medias_f1_score_seed))
    resultados["max_auc"].append(max(medias_auc_seed))
    resultados["max_acc_balanceada"].append(max(medias_acc_balanceada_seed))

    resultados["media_f1_score"].append(np.mean(medias_f1_score_seed))
    resultados["media_auc"].append(np.mean(medias_auc_seed))
    resultados["media_acc_balanceada"].append(np.mean(medias_acc_balanceada_seed))

    resultados["mediana_f1_score"].append(np.median(medias_f1_score_seed))
    resultados["mediana_auc"].append(np.median(medias_auc_seed))
    resultados["mediana_acc_balanceada"].append(np.median(medias_acc_balanceada_seed))

    for f in files:
        if f in r:
            resultado = pd.read_csv("../datasets/" + f + ".csv")

    instancias_erradas = []

    for i in range(len(dados)):
        previsoes = ast.literal_eval(dados["previsoes"][i])
        indices_instancias = ast.literal_eval(str(dados["indices"][i]))
        gabarito = list(resultado["Class"][indices_instancias])

        comparacao = []

        for g, j in zip(gabarito, previsoes):
            comparacao.append(g == j)

        instancias_erradas.append([indices_instancias[k] for k, j in enumerate(comparacao) if not j])

    frequencia_maxima = max(Counter(list(itertools.chain.from_iterable(instancias_erradas))).values())
    frequencia_instancias_erradas = list(Counter(list(itertools.chain.from_iterable(instancias_erradas))).items())
    instancias_erradas = [i[0] for i in frequencia_instancias_erradas if i[1] >= frequencia_maxima]
    arquivos_mais_classificados_erroneamente = list(resultado[resultado.columns[0]][instancias_erradas].values)

    resultados["instancias_mais_erradas"].append(arquivos_mais_classificados_erroneamente)

resultados = pd.DataFrame(resultados)

resultados.to_csv('../resultados/default/resumo_resultado_default.csv', index = False, encoding = 'utf-8', sep = ";")

  0%|          | 0/288 [00:00<?, ?it/s]

100%|██████████| 288/288 [00:22<00:00, 12.91it/s]


### Técnicas de Imputação

Foram aplicadas 4 técnicas de imputação: substituição pela média, KNN Imputer, remoção de features com valores ausentes e criação de colunas para identificar categorias de ELA que apresentavam valores faltantes. A análise abaixo separa as médias de f1 score obtidas por cada uma das combinações de acordo com a técnica de imputação empregada. Combinações sem etapa de imputação foram desconsideradas.

In [17]:
imputation_approaches = {"imputer_mean": [], "set_ela_error": [], "remove_missing_values": [], "knn_imputer": []}

In [18]:
lista = [(resultados.loc[i]["arquivo"], resultados.loc[i]["media_f1_score"]) for i, f in enumerate(resultados.values) if not "None" in resultados.loc[i]["arquivo"]]

In [19]:
for l in lista:
    for f in imputation_approaches:
        if f in l[0]:
            imputation_approaches[f].append(l[1])
    

#### Teste de Friedman
H0: não existe diferença significativa estatística entre as amostras (alpha = 0.05)

H1: existe diferença estatística significativa entre as amostras 

In [20]:
stats.friedmanchisquare(imputation_approaches["imputer_mean"], imputation_approaches["knn_imputer"], imputation_approaches["remove_missing_values"], imputation_approaches["set_ela_error"])

FriedmanchisquareResult(statistic=np.float64(41.36942675159236), pvalue=np.float64(5.459643500547492e-09))

#### Teste de Nemenyi
Para cada par de amostras, identifica se há diferença estatística (H1) ou não (H0).

0 - Substituição pela média

1 - KNN Imputer

2 - Remoção de features com valores ausentes

3 - Criação de categorias para identificar categorias ELA que apresentam valores faltantes

In [21]:
data = np.array([imputation_approaches["imputer_mean"], imputation_approaches["knn_imputer"], imputation_approaches["remove_missing_values"], imputation_approaches["set_ela_error"]])

In [22]:
sp.posthoc_nemenyi_friedman(data.T)

,0,1,2,3
0,1.000000e+00,0.606187,3.797728e-07,0.000027
1,6.061870e-01,1.000000,1.736535e-04,0.004416
2,3.797728e-07,0.000174,1.000000e+00,0.844289
3,2.664084e-05,0.004416,8.442891e-01,1.000000


In [23]:
buffer_imputation = []
for imputation in imputation_approaches:
    buffer_imputation.append(np.mean(imputation_approaches[imputation]))

In [24]:
buffer_imputation

[np.float64(0.7705865340003031),
 np.float64(0.7932508256458246),
 np.float64(0.7944649722709092),
 np.float64(0.7735403643432635)]

**Conclusão**: Remoção de features com valores ausentes tem o melhor desempenho e possui diferença significativa com as demais técnicas ou possui uma quantidade de features menor (redução de dimensão do dataset).

### Threshold de Correlação

Foram aplicadas 4 limiares de correlação: {0.8, 0.85, 0.9, 0.95}. A análise abaixo separa as médias de f1 score obtidas por cada uma das combinações de acordo com o threshold de correlação empregado.

In [25]:
lista_arquivos_flacco_sem_remove_missing_values = [resultados.loc[i]["arquivo"] for i, f in enumerate(resultados.values) if "ela_features_flacco" in resultados.loc[i]["arquivo"] and not "remove_missing_values" in resultados.loc[i]["arquivo"]]

In [26]:
lista = [(resultados.loc[i]["arquivo"], resultados.loc[i]["media_f1_score"]) for i, f in enumerate(resultados.values) if not resultados.loc[i]["arquivo"] in lista_arquivos_flacco_sem_remove_missing_values]

In [27]:
corr_thresholds = {"0_8.": [], "0_85": [], "0_9.": [], "0_95": []}

In [28]:
for l in lista:
    for f in corr_thresholds:
        if f in l[0]:
            corr_thresholds[f].append(l[1])
    

**Conclusão**: Não existe diferença significativa entre os limiares de correlação aplicados. Portanto, adota-se o threshold de 0.8 como a melhor opção, visto que remove a maior quantidade possível de features sem afetar o desempenho final do modelo.

#### Teste de Friedman

In [29]:
stats.friedmanchisquare(corr_thresholds["0_8."], corr_thresholds["0_85"], corr_thresholds["0_9."], corr_thresholds["0_95"])

FriedmanchisquareResult(statistic=np.float64(1.5913043478261224), pvalue=np.float64(0.6613631039707857))

In [30]:
buffer_corr = []
for corr in corr_thresholds:
    buffer_corr.append(np.mean(corr_thresholds[corr]))

In [31]:
buffer_corr

[np.float64(0.7842485327120722),
 np.float64(0.7854352085131303),
 np.float64(0.7885686124415616),
 np.float64(0.7919848981618983)]

### Algoritmos de Machine Learning (ML)

Foram aplicados 8 algoritmos de ML como meta-learnes: 1. Naive Bayes (NB), 2. Decision Tree, 3. K-Nearest Neighbors (kNN), 4. Random Forest (RF), 5. SVM com kernel RBF (SVM-RBF), 6. SVM com kernel linear (SVM-Lin), 7. Regressão Logística (RL) e 8. eXtreme Gradient Boosting (XGB). A análise abaixo separa as médias de f1 score obtidas por cada uma das combinações de acordo com o algoritmo empregado.

In [32]:
lista = [(resultados.loc[i]["arquivo"], resultados.loc[i]["media_f1_score"]) for i, f in enumerate(resultados.values) if not resultados.loc[i]["arquivo"] in lista_arquivos_flacco_sem_remove_missing_values and "0_8." in resultados.loc[i]["arquivo"]]

In [33]:
algorithms = {"NB": [], "DT":[], "KNN": [], "RF": [], "SVM_RBF": [], "SVM_LIN": [], "LogisticRegression": [], "XGBoost": []}

In [34]:
for l in lista:
    for f in algorithms:
        if f in l[0]:
            algorithms[f].append(l[1])
    

#### Teste de Friedman

In [35]:
stats.friedmanchisquare(algorithms["NB"], algorithms["DT"], algorithms["KNN"], algorithms["RF"], algorithms["SVM_RBF"], algorithms["SVM_LIN"], algorithms["LogisticRegression"], algorithms["XGBoost"])

FriedmanchisquareResult(statistic=np.float64(14.333333333333329), pvalue=np.float64(0.04556053530040479))

In [36]:
data = np.array([algorithms["NB"], algorithms["DT"], algorithms["KNN"], algorithms["RF"], algorithms["SVM_RBF"], algorithms["SVM_LIN"], algorithms["LogisticRegression"], algorithms["XGBoost"]])

#### Teste de Nemenyi Post Hoc

In [37]:
sp.posthoc_nemenyi_friedman(data.T)

,0,1,2,3,4,5,6,7
0,1.000000,0.999978,1.000000,0.597130,0.195221,0.807805,0.708976,0.886305
1,0.999978,1.000000,0.999664,0.372133,0.086902,0.597130,0.481766,0.708976
2,1.000000,0.999664,1.000000,0.708976,0.275295,0.886305,0.807805,0.941347
3,0.597130,0.372133,0.708976,1.000000,0.997811,0.999978,1.000000,0.999664
4,0.195221,0.086902,0.275295,0.997811,1.000000,0.974578,0.991243,0.941347
5,0.807805,0.597130,0.886305,0.999978,0.974578,1.000000,1.000000,1.000000
6,0.708976,0.481766,0.807805,1.000000,0.991243,1.000000,1.000000,0.999978
7,0.886305,0.708976,0.941347,0.999664,0.941347,1.000000,0.999978,1.000000


In [38]:
buffer_al = []
for al in algorithms:
    buffer_al.append(np.mean(algorithms[al]))

In [39]:
buffer_al

[np.float64(0.588745442970929),
 np.float64(0.7613266198727678),
 np.float64(0.7820595056987362),
 np.float64(0.8281102431764102),
 np.float64(0.8370185816737542),
 np.float64(0.825381205678657),
 np.float64(0.8305984861456125),
 np.float64(0.8207481764797105)]

**Conclusão**: Os algoritmos RF, SVM-RBF, SVM-Lin, LR e XGB apresentam uma média de F1 score entre 0.82 e 0.83, sem diferença significativa. Portanto, adota-se o algoritmo de ML com menor desvio padrão na métrica de interesse, a fim de garantir menor variabilidade nos resultados obtidos.

### Meta-dataset

Foram utilizados três meta-datasets para alimentar o sistema de meta-learning: 1. meta-features clássicas de ELA, 2. meta-features extraídas da biblioteca pyMFE e 3. combinação das meta-features 1 e 2.

In [40]:
lista = [(resultados.loc[i]["arquivo"], resultados.loc[i]["media_f1_score"]) for i, f in enumerate(resultados.values) if not resultados.loc[i]["arquivo"] in lista_arquivos_flacco_sem_remove_missing_values and "0_8." in resultados.loc[i]["arquivo"] and "SVM_RBF" in resultados.loc[i]["arquivo"]]

#### Teste de Friedman

In [42]:
stats.friedmanchisquare(lista[0][1], lista[1][1], lista[2][1])

FriedmanchisquareResult(statistic=np.float64(2.0), pvalue=np.float64(0.36787944117144245))

**Conclusão**: Não existe diferença significativa entre os meta-datasets selecionados em relação à métrica f1 score. Portanto, adota-se como a melhor escolha o meta-dataset com meta-features ELA, o qual apresenta somente 36 meta-features.